In [1]:
import torch
import torch.nn as nn

In [28]:
text="I love python Programming"
tokens=text.lower().split()
print("Tokens:",tokens)

Tokens: ['i', 'love', 'python', 'programming']


In [ ]:
vocab=sorted(set(tokens))
print("Vocabulary:",vocab)

wordtoid={word:id for id,word in enumerate(vocab)}
print("Word To ID:",wordtoid)

idtoword={id:word for word,id in wordtoid.items()}
print("ID To Word:",idtoword)

Vocabulary: ['i', 'love', 'programming', 'python']
Word To ID: {'i': 0, 'love': 1, 'programming': 2, 'python': 3}
ID To Word: {0: 'i', 1: 'love', 2: 'programming', 3: 'python'}


In [30]:
token_ids=[wordtoid[i] for i in tokens]
print("Token IDS:",token_ids)

Token IDS: [0, 1, 3, 2]


In [31]:
# Create Training Data
input_ids=token_ids[:-1]
target_ids=token_ids[1:]
print("INPUT IDS:",input_ids)
print("TARGET IDS:",target_ids)

INPUT IDS: [0, 1, 3]
TARGET IDS: [1, 3, 2]


In [32]:
input_tensor=torch.tensor([input_ids],dtype=torch.long)
target_tensor=torch.tensor([target_ids],dtype=torch.long)

print("Input Tensor:",input_tensor)
print("Target Tensor:",target_tensor)
print("Input Shape:",input_tensor.shape)
print("Target Shape:",target_tensor.shape)

Input Tensor: tensor([[0, 1, 3]])
Target Tensor: tensor([[1, 3, 2]])
Input Shape: torch.Size([1, 3])
Target Shape: torch.Size([1, 3])


In [33]:
class RNNModel(nn.Module):
    def __init__(self,vocabularysize,embeddingdim,hiddensize):
        super().__init__()

        self.embedding=nn.Embedding(vocabularysize,embeddingdim)
        self.rnn = nn.RNN(input_size=embeddingdim,hidden_size=hiddensize,batch_first=True)
        self.linear=nn.Linear(hiddensize,vocabularysize)

    def forward(self,x):
        embeddings= self.embedding(x)
        output,hidden=self.rnn(embeddings)
        output=self.linear(output)
        return output

In [34]:
vocabularysize=len(vocab)
embeddingdim=10
hiddensize=20
model=RNNModel(vocabularysize,embeddingdim,hiddensize)
lossfunction=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.01)

In [35]:
for epoch in range(1000):
    output=model(input_tensor)
    output=output.reshape(-1,vocabularysize)
    target=target_tensor.reshape(-1)
    loss=lossfunction(output,target)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0:
        print('Epoch:',epoch,'Loss:',loss.item())


Epoch: 0 Loss: 1.1863163709640503
Epoch: 100 Loss: 0.0019245855510234833
Epoch: 200 Loss: 0.0009139119647443295
Epoch: 300 Loss: 0.0005398306529968977
Epoch: 400 Loss: 0.00036054220981895924
Epoch: 500 Loss: 0.0002596036356408149
Epoch: 600 Loss: 0.00019663597049657255
Epoch: 700 Loss: 0.00015436393732670695
Epoch: 800 Loss: 0.0001245260937139392
Epoch: 900 Loss: 0.00010263385047437623


In [36]:
model.eval()

RNNModel(
  (embedding): Embedding(4, 10)
  (rnn): RNN(10, 20, batch_first=True)
  (linear): Linear(in_features=20, out_features=4, bias=True)
)

In [38]:
text="i love python"
tokens=text.lower().split()
texttoid=[wordtoid[i] for i in tokens]
input_tensor=torch.tensor(texttoid,dtype=torch.long)

print(input_tensor.shape)
with torch.no_grad():

    output = model(input_tensor)
    print(output.shape)

    # Take the output of the LAST word
    # last_output = output[:, -1, :] # if batch
    last_output = output[-1, :] # not batch

    # Find the word with highest score
    predicted_id = torch.argmax(
        last_output,
        dim=-1
    )

predicted_word = idtoword[
    predicted_id.item()
]
print("Next word:",text, predicted_word)

torch.Size([3])
torch.Size([3, 4])
Next word: i love python programming


### Problem of RNN

- Struggles to learn Long sequences relationships
- Gradients can be extremely small called vanishing gradient problem.
- Gradients can become extremely large called Exploding gradient
- Sequential processing is slow


In [ ]:
# NEXT LSTM